In [ ]:
from pathlib import Path
import shutil
import subprocess
from praatio import textgrid
import os

EXCERPT_ID = "excerpt_000"

PROJECT_ROOT = Path("/workspace/storyfx")

# ============================================================
# MFA ENVIRONMENT
# ============================================================

MAMBA_ROOT_PREFIX = "/workspace/micromamba"
MFA_ROOT_DIR = "/workspace/mfa"

os.environ["MAMBA_ROOT_PREFIX"] = MAMBA_ROOT_PREFIX
os.environ["MFA_ROOT_DIR"] = MFA_ROOT_DIR

MFA_ENV = {
    **os.environ,
    "MAMBA_ROOT_PREFIX": MAMBA_ROOT_PREFIX,
    "MFA_ROOT_DIR": MFA_ROOT_DIR,
}

# ============================================================
# DATA DIRECTORIES
# ============================================================

DATA_DIR = PROJECT_ROOT / "data"
EXCERPT_DIR = DATA_DIR / EXCERPT_ID

RAW_DIR = EXCERPT_DIR / "raw"
ALIGNMENT_DIR = EXCERPT_DIR / "alignment"

# ============================================================
# MFA WORKING DIRECTORIES
# ============================================================

MFA_DIR = PROJECT_ROOT / "scripts" / "MFA"

CORPUS_DIR = MFA_DIR / "corpus"
MFA_OUTPUT_DIR = MFA_DIR / "aligned_output"

# ============================================================
# RAW INPUT FILES
# ============================================================

AUDIO_FILE = RAW_DIR / f"{EXCERPT_ID}.wav"
PRAAT_TEXTGRID = RAW_DIR / f"{EXCERPT_ID}.TextGrid"

# ============================================================
# FINAL OUTPUT FILES
# ============================================================

FINAL_TEXTGRID = (
    ALIGNMENT_DIR
    / f"{EXCERPT_ID}_fa.TextGrid"
)

FINAL_PARSED = (
    ALIGNMENT_DIR
    / f"{EXCERPT_ID}_fa_parsed.txt"
)

print("Project root:", PROJECT_ROOT)
print("Mamba root:", MAMBA_ROOT_PREFIX)
print("MFA model root:", MFA_ROOT_DIR)

In [1]:
print("Project root:", PROJECT_ROOT)
print("Excerpt:", EXCERPT_ID)

print("\nMFA executable:")
print(MFA_EXECUTABLE)

print("\nMFA inputs:")
print("Audio:", AUDIO_FILE)
print("Praat TextGrid:", PRAAT_TEXTGRID)

print("\nFinal outputs:")
print("Aligned TextGrid:", FINAL_TEXTGRID)
print("Parsed FA:", FINAL_PARSED)


if not MFA_EXECUTABLE.exists():
    raise FileNotFoundError(
        f"MFA installation not found: {MFA_EXECUTABLE}"
    )

subprocess.run(
    [
        str(MFA_EXECUTABLE),
        "version"
    ],
    check=True,
)

NameError: name 'PROJECT_ROOT' is not defined

In [7]:
# Validate input files
if not AUDIO_FILE.exists():
    raise FileNotFoundError(
        f"Missing WAV file: {AUDIO_FILE}"
    )

if not PRAAT_TEXTGRID.exists():
    raise FileNotFoundError(
        f"Missing Praat TextGrid: {PRAAT_TEXTGRID}"
    )


# Clean temporary MFA directories

if CORPUS_DIR.exists():
    shutil.rmtree(CORPUS_DIR)

if MFA_OUTPUT_DIR.exists():
    shutil.rmtree(MFA_OUTPUT_DIR)


# Recreate directories

CORPUS_DIR.mkdir(parents=True, exist_ok=True)
MFA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALIGNMENT_DIR.mkdir(parents=True, exist_ok=True)


# Copy WAV + manually segmented Praat TextGrid into MFA corpus

shutil.copy2(
    AUDIO_FILE,
    CORPUS_DIR / f"{EXCERPT_ID}.wav"
)

shutil.copy2(
    PRAAT_TEXTGRID,
    CORPUS_DIR / f"{EXCERPT_ID}.TextGrid"
)


print("MFA corpus prepared successfully.")

print(
    CORPUS_DIR / f"{EXCERPT_ID}.wav"
)

print(
    CORPUS_DIR / f"{EXCERPT_ID}.TextGrid"
)

MFA corpus prepared successfully.
C:\Users\gorda\Desktop\StoryFX\storyfx\scripts\MFA\corpus\excerpt_000.wav
C:\Users\gorda\Desktop\StoryFX\storyfx\scripts\MFA\corpus\excerpt_000.TextGrid


In [8]:
def run_mfa(*args):
    return subprocess.run(
        [
            "micromamba",
            "run",
            "-r",
            MAMBA_ROOT_PREFIX,
            "-n",
            "mfa_env",
            "mfa",
            *args,
        ],
        env=MFA_ENV,
        check=True,
    )

MFA alignment complete.


In [ ]:
run_mfa(
    "align",
    str(CORPUS_DIR),
    "english_us_arpa",
    "english_us_arpa",
    str(MFA_OUTPUT_DIR),
    "--clean",
)

print("MFA alignment complete.")

In [9]:
MFA_TEXTGRID = MFA_OUTPUT_DIR / f"{EXCERPT_ID}.TextGrid"

if not MFA_TEXTGRID.exists():
    raise FileNotFoundError(
        f"Expected MFA output not found: {MFA_TEXTGRID}"
    )

shutil.copy2(
    MFA_TEXTGRID,
    FINAL_TEXTGRID
)

print("Saved:", FINAL_TEXTGRID)

Saved: C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\alignment\excerpt_000_fa.TextGrid


In [ ]:
tg = textgrid.openTextgrid(
    str(FINAL_TEXTGRID),
    includeEmptyIntervals=False,
)

word_tier = tg.getTier("words")

parsed_words = []

for entry in word_tier.entries:
    start, end, word = entry

    word = word.strip()

    if word:
        parsed_words.append(
            f"{word}|{end:.2f}"
        )

parsed_text = " ".join(parsed_words)

print(parsed_text)

In [13]:
FINAL_PARSED.write_text(
    parsed_text,
    encoding="utf-8",
)

print("Saved:", FINAL_PARSED)

Saved: C:\Users\gorda\Desktop\StoryFX\storyfx\data\excerpt_000\alignment\excerpt_000_fa_parsed.txt
